In [ ]:
import pandas as pd
import numpy as np
import torch
import os
import sys
from tqdm import tqdm, trange

sys.path.append("../../")
import biked_commons
from biked_commons.design_evaluation.design_evaluation import *
from biked_commons.resource_utils import split_datasets_path
from biked_commons.conditioning import conditioning
from biked_commons.design_evaluation.scoring import *

In [ ]:
data = pd.read_csv(split_datasets_path("bike_bench.csv"), index_col=0)

#sample 100
data = data.sample(100, random_state=0)
data_tens = torch.tensor(data.values, dtype=torch.float32)

evaluator, requirement_names, requirement_types = construct_tensor_evaluator(StandardEvaluations, data.columns)

def calc_composite_score(data_tens, evaluator, requirement_names, requirement_types):
    # Calculate the composite score for each row in the data tensor
    composite_scores = []
    for i in trange(data_tens.shape[0], desc="Calculating composite scores"):
        row = data_tens[i].unsqueeze(0)
        scores = evaluator(row, requirement_names, requirement_types)
        composite_score = torch.mean(scores).item()
        composite_scores.append(composite_score)
    return